In [3]:
!pip install "numpy<2"

Defaulting to user installation because normal site-packages is not writeable
  Using cached numpy-1.26.4-cp312-cp312-win_amd64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp312-cp312-win_amd64.whl (15.5 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 2.5.0
    Uninstalling numpy-2.5.0:
      Successfully uninstalled numpy-2.5.0


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
mlxtend 0.25.0 requires numpy>=2.3.5, but you have numpy 1.26.4 which is incompatible.
scipy 1.18.0 requires numpy<2.8,>=2.0.0, but you have numpy 1.26.4 which is incompatible.
streamlit 1.32.0 requires pandas<3,>=1.3.0, but you have pandas 3.0.3 which is incompatible.


In [4]:
!pip install --upgrade numpy scipy statsmodels pandas

Defaulting to user installation because normal site-packages is not writeable
  Using cached numpy-2.5.0-cp312-cp312-win_amd64.whl.metadata (6.6 kB)
Using cached numpy-2.5.0-cp312-cp312-win_amd64.whl (12.4 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
numba 0.59.1 requires numpy<1.27,>=1.22, but you have numpy 2.5.0 which is incompatible.
pywavelets 1.5.0 requires numpy<2.0,>=1.22.4, but you have numpy 2.5.0 which is incompatible.
streamlit 1.32.0 requires numpy<2,>=1.19.3, but you have numpy 2.5.0 which is incompatible.
streamlit 1.32.0 requires pandas<3,>=1.3.0, but you have pandas 3.0.3 which is incompatible.


In [5]:
!pip install --upgrade numexpr bottleneck

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/160.2 kB ? eta -:--:--
   -- ------------------------------------- 10.2/160.2 kB ? eta -:--:--
   ------- ------------------------------- 30.7/160.2 kB 325.1 kB/s eta 0:00:01
   -------------------------------------- - 153.6/160.2 kB 1.3 MB/s eta 0:00:01
   ---------------------------------------- 160.2/160.2 kB 1.2 MB/s eta 0:00:00
   ---------------------------------------- 0.0/113.5 kB ? eta -:--:--
   ---------------------------------------- 113.5/113.5 kB 3.3 MB/s eta 0:00:00


In [9]:
import pandas as pd
import numpy as np

# Loading the dataset
df = pd.read_csv('mktmix.csv')

# Displaying the shape of the dataset (rows, columns)
print(f"Dataset Shape: {df.shape}\n")

# Checking data types and non-null counts
print("--- Data Info ---")
df.info()

# Counting exactly how many missing values are in each column
print("\n--- Missing Values ---")
print(df.isnull().sum())

Dataset Shape: (104, 9)

--- Data Info ---
<class 'pandas.DataFrame'>
RangeIndex: 104 entries, 0 to 103
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   NewVolSales        104 non-null    int64  
 1   Base_Price         104 non-null    float64
 2   Radio              100 non-null    float64
 3   InStore            104 non-null    float64
 4   NewspaperInserts   6 non-null      str    
 5   Discount           104 non-null    float64
 6   TV                 104 non-null    float64
 7   Stout              104 non-null    float64
 8   Website_Campaign   14 non-null     str    
dtypes: float64(6), int64(1), str(2)
memory usage: 7.4 KB

--- Missing Values ---
NewVolSales           0
Base_Price            0
Radio                 4
InStore               0
NewspaperInserts     98
Discount              0
TV                    0
Stout                 0
Website_Campaign     90
dtype: int64


In [11]:
df.columns = df.columns.str.strip()

# Filling missing 'Radio' values with the average Radio spend
df['Radio'] = df['Radio'].fillna(df['Radio'].mean())

# Converting text columns to binary flags (1 = Campaign active, 0 = No campaign)
df['Newspaper_Flag'] = df['NewspaperInserts'].notna().astype(int)
df['Website_Flag'] = df['Website_Campaign'].notna().astype(int)

# Droping the old text columns since our model only wants numbers
df = df.drop(columns=['NewspaperInserts', 'Website_Campaign'])

# Verifying the cleanup
print("--- Missing Values After Cleanup ---")
print(df.isnull().sum())

print("\n--- First 5 Rows of Cleaned Data ---")
print(df.head())

--- Missing Values After Cleanup ---
NewVolSales       0
Base_Price        0
Radio             0
InStore           0
Discount          0
TV                0
Stout             0
Newspaper_Flag    0
Website_Flag      0
dtype: int64

--- First 5 Rows of Cleaned Data ---
   NewVolSales  Base_Price  Radio  InStore  Discount          TV    Stout  \
0        19564   15.029276  245.0   15.452     0.000  101.780000  2.28342   
1        19387   15.029276  314.0   16.388     0.000   76.734000  2.22134   
2        23889   14.585093  324.0   62.692     0.050  131.590200  2.00604   
3        20055   15.332887  298.0   16.573     0.000  119.627060  2.19897   
4        20064   15.642632  279.0   41.504     0.045  103.438118  1.81860   

   Newspaper_Flag  Website_Flag  
0               0             0  
1               0             0  
2               0             0  
3               0             0  
4               0             0  


In [15]:
import numpy as np

# Defining the Adstock Function
def apply_adstock(x, L, theta, delay=0):
    """
    Applies the adstock transformation.
    x: Original spend array
    L: Maximum lag period (we'll use 4 weeks)
    theta: Decay weight (0 < theta < 1)
    """
    x = np.array(x)
    adstocked_x = np.zeros(len(x))
    
    for t in range(len(x)):
        # Calculate the carryover effect from previous weeks up to the maximum lag
        for l in range(L):
            if t - l - delay >= 0:
                adstocked_x[t] += x[t - l - delay] * (theta ** l)
    return adstocked_x

# Defining the Saturation Function
def apply_saturation(x, alpha):
    """
    Applies the exponential saturation curve.
    x: Adstocked spend array
    alpha: Saturation parameter (controls how fast returns diminish)
    """
    return 1 - np.exp(-alpha * x)

# Applying transformations to our Media Channels
# In a real advanced MMM, we would use an optimization algorithm to find the absolute PERFECT theta and alpha for each channel
# For this project, we will be assigning standard industry estimates

# TV: High carryover (people remember TV ads), low saturation (broad audience)
df['TV_Adstock'] = apply_adstock(df['TV'], L=4, theta=0.6)
df['TV_Transformed'] = apply_saturation(df['TV_Adstock'], alpha=0.005)

# Radio: Medium carryover, medium saturation
df['Radio_Adstock'] = apply_adstock(df['Radio'], L=4, theta=0.4)
df['Radio_Transformed'] = apply_saturation(df['Radio_Adstock'], alpha=0.01)

# InStore: Low carryover (immediate effect), high saturation (limited foot traffic)
df['InStore_Adstock'] = apply_adstock(df['InStore'], L=4, theta=0.2)
df['InStore_Transformed'] = apply_saturation(df['InStore_Adstock'], alpha=0.02)

# Verifying the new columns were added
print(df[['TV', 'TV_Adstock', 'TV_Transformed']].head())

           TV  TV_Adstock  TV_Transformed
0  101.780000   101.78000        0.398844
1   76.734000   137.80200        0.497927
2  131.590200   214.27140        0.657457
3  119.627060   248.18990        0.710890
4  103.438118   239.16137        0.697540


In [17]:
import statsmodels.api as sm

# Defining our target variable (What we want to predict)
y = df['NewVolSales']

# Defining our independent variables (What drives the sales)
# We use our *Transformed* media variables, not the raw spend!
X = df[['Base_Price', 'Discount', 'TV_Transformed', 
        'Radio_Transformed', 'InStore_Transformed', 
        'Newspaper_Flag', 'Website_Flag']]

# Adding a constant 
# (This represents our 'Base Sales' - what we would sell if we spent $0 on marketing)
X = sm.add_constant(X)

# Fitting the OLS model
mmm_model = sm.OLS(y, X).fit()

# Printing the statistical summary
print(mmm_model.summary())

                            OLS Regression Results                            
Dep. Variable:            NewVolSales   R-squared:                       0.683
Model:                            OLS   Adj. R-squared:                  0.660
Method:                 Least Squares   F-statistic:                     29.55
Date:                Thu, 25 Jun 2026   Prob (F-statistic):           2.32e-21
Time:                        23:58:24   Log-Likelihood:                -853.21
No. Observations:                 104   AIC:                             1722.
Df Residuals:                      96   BIC:                             1744.
Df Model:                           7                                         
Covariance Type:            nonrobust                                         
                          coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------
const                4.899e+04   2

In [19]:
# ROI CALCULATION

# Extracting the TV Coefficient from the model
tv_coef = mmm_model.params['TV_Transformed']

# Calculating Incremental Volume driven by TV
# (Coefficient * Transformed Media)
df['TV_Incremental_Volume'] = tv_coef * df['TV_Transformed']

# Calculating Incremental Revenue
# (Extra volume sold * the price of the toy that week)
df['TV_Incremental_Revenue'] = df['TV_Incremental_Volume'] * df['Base_Price']

# Calculating Total ROI (Return on Ad Spend)
total_tv_revenue = df['TV_Incremental_Revenue'].sum()
total_tv_spend = df['TV'].sum() # We use the RAW money spent here!

tv_roi = total_tv_revenue / total_tv_spend

# Printing the final business report
print("=== FINAL BUSINESS DELIVERABLE ===")
print(f"Total Raw TV Spend: ${total_tv_spend:,.2f}")
print(f"Total Incremental Revenue from TV: ${total_tv_revenue:,.2f}")
print(f"TV ROI (ROAS): ${tv_roi:.2f} for every $1 spent")

=== FINAL BUSINESS DELIVERABLE ===
Total Raw TV Spend: $14,665.02
Total Incremental Revenue from TV: $4,778,333.00
TV ROI (ROAS): $325.83 for every $1 spent
